# CaRS-50 — download & preprocess

*Swales CARS moves in research-article introductions (3 or 11 classes)*

**What it is.** 50 BioRxiv article introductions, annotated sentence by sentence with Swales' CARS Move and Step scheme. **The annotators themselves reached only κ ≈ 0.43** — so on this track, "the model is wrong" and "the scheme is fuzzy" are both live explanations, and telling them apart is the interesting part.

**Difficulty of the labeling judgment:** ★★★ — hard. Judging moves in an introduction needs more context than a single sentence gives you.

**Licence:** CC BY 4.0  
**Cite:** Lam, C. & Nnamoko, N. (2025). *Mendeley Data*, V1. doi:10.17632/kwr9s5c4nk.1

---

Every dataset in this course is reshaped into the **same canonical schema**, so one pipeline works for all of them:

```json
[{"id": 1, "text": "...", "label": "..."}]
```

The *raw* data, though, looks different every time. **That difference is the lesson** — half of building a gold standard is getting messy real data into a clean, consistent shape.

> This notebook is **generated** from `scripts/reshape.py`. The reshaping code below is the same code `scripts/prep_datasets.py` runs — not a copy of it. If you want to change how the data is reshaped, edit `reshape.py` and re-run `scripts/_generate_download_notebooks.py`.

## Step 1 — Download the raw data

This one is on **Mendeley Data**, which has a public API. We ask it for the dataset's file list, then download each file. The CDN refuses requests that do not look like a browser, hence the `User-Agent` header.

In [ ]:
import json, urllib.request, pathlib

RAW_DIR = pathlib.Path("cars50")
RAW_DIR.mkdir(exist_ok=True)

def fetch(url):
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    return urllib.request.urlopen(request, timeout=60)

meta = json.loads(fetch("https://data.mendeley.com/public-api/datasets/kwr9s5c4nk").read())
for record in meta["files"]:
    target = RAW_DIR / record["filename"]
    if not target.exists():
        target.write_bytes(fetch(record["content_details"]["download_url"]).read())
print("downloaded", len(list(RAW_DIR.glob("*.xml"))), "XML files")

## Step 2 — Look at the raw format

**XML** this time. Each sentence carries a `step` code like `1b`:

```xml
<sentence><sentenceID/><text/><step>1b</step></sentence>
```

In [ ]:
print(open(sorted(RAW_DIR.glob("*.xml"))[0], encoding="utf-8").read()[:900])

## Step 3 — Reshape into the canonical schema

The interesting decision here is **granularity**, and one parse gives you both:

- the leading digit of `1b` is the **Move** → 3 classes, a fair task;
- the whole code `1b` is the **Step** → 11 classes, the stretch version.

Which one you pick is a scheme decision, and it belongs in your `PLAN.md`. Sentences with no code, or a code that does not start with a move digit, are dropped.

In [ ]:
def reid(items):
    """Renumber ids sequentially from 1, keeping the current order."""
    renumbered = []
    next_id = 1
    for item in items:
        new_item = dict(item)
        new_item["id"] = next_id
        renumbered.append(new_item)
        next_id = next_id + 1
    return renumbered

def reshape_cars50(cars50_dir):
    """Parse the 50 XML introductions into TWO datasets: moves, and move+step.

    XML shape:
        <sentence><sentenceID/><text/><step>1b</step></sentence>

    The `step` code is like "1b": the leading DIGIT is the Move, the whole code is the
    Step. So one parse gives two granularities, and which you use is a scheme decision:
    3 classes is a fair task, 11 classes is the stretch version. Returns
    (move_rows, step_rows).
    """
    source_dir = Path(cars50_dir)
    move_rows = []
    step_rows = []
    for xml_path in sorted(source_dir.glob("*.xml")):
        tree = ET.parse(xml_path)
        for sentence in tree.iter("sentence"):
            text_element = sentence.find("text")
            step_element = sentence.find("step")
            if text_element is None or step_element is None:
                continue
            text = (text_element.text or "").strip()
            code = (step_element.text or "").strip()
            # Skip anything unlabelled, or whose code does not start with a move digit.
            if not text or not code or not code[0].isdigit():
                continue
            move_rows.append({"id": 0, "text": text, "label": "Move " + code[0]})
            step_rows.append({"id": 0, "text": text, "label": code})
    return reid(move_rows), reid(step_rows)

In [ ]:
move_rows, step_rows = reshape_cars50(RAW_DIR)

rows = move_rows          # 3 classes. Swap in step_rows for the 11-class version.
print("moves:", len(move_rows), " steps:", len(step_rows))

## Step 4 — Check the label balance

In [ ]:
from collections import Counter

print("total items:", len(rows))
print("label counts:", dict(Counter(item["label"] for item in rows)))
rows[:3]        # peek at the first three reshaped items

## A note on what you just built

This is the **pool** — everything usable in the corpus, with its natural label imbalance intact. It is *not* your gold set.

Your gold set comes next, in the project notebook: `sample_pool` draws a *balanced* subset from this pool (equal items per label), which is what makes precision, recall, F1 and the confusion matrix readable. Keeping the two separate also leaves the unsampled items free to serve as few-shot examples without leaking the answers you are testing on.

So: build the pool once, here. Sample from it there.

## Step 5 — Save it

In [ ]:
# Save the pool. Two places you might want it:
#   * this repo, if you cloned it:  "../data/pools/cars50_pool.json"
#   * your Google Drive, so it survives the Colab runtime resetting
import json

OUT_FILE = "cars50_pool.json"

# In Colab, uncomment these two lines to write straight to your Drive:
# from google.colab import drive; drive.mount("/content/drive")
# OUT_FILE = "/content/drive/MyDrive/cars50_pool.json"

with open(OUT_FILE, "w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)
print("Saved", len(rows), "items to", OUT_FILE)

# For the 11-class version, set rows = step_rows above and save as cars50_step_pool.json.